# LEAF — Local Explanation evAluation Framework

This notebook computes all **five LEAF metrics** defined in Amparore et al. (2021)
for the LIME and SHAP explanations generated on our MLP black-box classifier.

| Metric | What it measures |
|---|---|
| **Conciseness** | Number K of features in the explanation (set by the user) |
| **Local Fidelity** | How well the white-box g mimics the black-box f in the neighbourhood N(x) |
| **Local Concordance** | How well g(x) matches f(x) for the single explained instance x |
| **Reiteration Similarity** | Stability of the feature set across R repeated runs |
| **Prescriptivity** | How useful g is for finding x' that crosses the decision boundary |


In [1]:
from pathlib import Path
import json

import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.linear_model import Ridge
import shap

/home/fateee/Desktop/ulyana/Implementing_a_Local_Explanations_Assessment_Framework/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED       = 42
K          = 10
R          = 30
N_PERTURB  = 1000
NUM_SAMPLES = 2000

DATA_DIR = Path("data")
MLP_DIR  = Path("artifacts/mlp")
LIME_DIR = Path("artifacts/lime")
SHAP_DIR = Path("artifacts/shap")
OUT_DIR  = Path("artifacts/leaf")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_test  = np.load(DATA_DIR / "X_test.npy")
y_test  = np.load(DATA_DIR / "y_test.npy")

meta          = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
feature_names = meta["feature_names"]
n_features    = len(feature_names)

print(f"Train: {X_train.shape}, Test: {X_test.shape}, Features: {n_features}")

Train: (341, 30), Test: (114, 30), Features: 30


In [4]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, 1),
        )

    def forward(self, x):
        return self.net(x)


class MLPWrapper:
    """Sklearn-compatible wrapper — exposes predict_proba for LIME / SHAP."""

    def __init__(self, mlp_model, device):
        self.model = mlp_model
        self.device = device

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            t = torch.tensor(np.asarray(X, dtype=np.float32)).to(self.device)
            probs1 = torch.sigmoid(self.model(t)).cpu().numpy().flatten()
        return np.column_stack([1.0 - probs1, probs1])


config = json.loads((MLP_DIR / "model_config.json").read_text(encoding="utf-8"))
device = "cuda" if torch.cuda.is_available() else "cpu"
_mlp = MLP(config["input_dim"]).to(device)
_mlp.load_state_dict(torch.load(MLP_DIR / "model_state.pt", map_location=device))
model = MLPWrapper(_mlp, device)


def f_predict(X):
    """Black-box prediction: probability of benign class."""
    return model.predict_proba(np.asarray(X, dtype=np.float32))[:, 1].astype(np.float64)


print("Model loaded.")

Model loaded.


In [5]:
lime_summary = json.loads((LIME_DIR / "lime_summary.json").read_text(encoding="utf-8"))
lime_map     = {rec["test_index"]: rec for rec in lime_summary["results"]}

shap_values    = np.load(SHAP_DIR / "shap_values_class1.npy")
expected_value = json.loads((SHAP_DIR / "expected_value.json").read_text(encoding="utf-8"))["expected_value"]

chosen       = json.loads((LIME_DIR / "chosen_test_indices.json").read_text(encoding="utf-8"))
test_indices = [int(i) for i in chosen["chosen_test_indices"]]

print(f"Loaded {len(test_indices)} test instances.")

Loaded 10 test instances.


## LEAF Metric Definitions

All five functions below follow the paper's definitions exactly.

In [6]:
def local_fidelity(f_fn, g_fn, x, X_train, n_samples=1000, seed=0):
    rng   = np.random.default_rng(seed)
    sigma = np.std(X_train, axis=0) + 1e-9
    Z     = x + rng.normal(0, sigma, size=(n_samples, len(x)))

    f_bin = (f_fn(Z) >= 0.5).astype(int)
    g_bin = (np.asarray(g_fn(Z), dtype=np.float64) >= 0.5).astype(int)

    return float(f1_score(f_bin, g_bin, zero_division=0))

In [7]:
def local_concordance(f_x, g_x):
    return float(max(0.0, 1.0 - abs(float(f_x) - float(g_x))))

In [8]:
def jaccard(set_a, set_b):
    union = set_a | set_b
    if not union:
        return 1.0
    return len(set_a & set_b) / len(union)


def reiteration_similarity(feature_sets):
    """Average pairwise Jaccard over a list of feature-index sets."""
    n = len(feature_sets)
    if n < 2:
        return 1.0
    scores = []
    for i in range(n):
        for j in range(i + 1, n):
            scores.append(jaccard(feature_sets[i], feature_sets[j]))
    return float(np.mean(scores))

In [9]:
def prescriptivity(f_fn, weights, intercept, k_indices, x, y_prime=0.5):
    """Compute LEAF prescriptivity score."""
    weights   = np.asarray(weights,   dtype=np.float64)
    k_indices = np.asarray(k_indices, dtype=np.intp)

    g_x = float(intercept) + float(np.dot(weights, x[k_indices]))

    w_norm_sq = float(np.dot(weights, weights))
    if w_norm_sq < 1e-10:
        return 0.0

    h_k = (y_prime - g_x) * weights / w_norm_sq

    x_prime = x.astype(np.float64)
    x_prime[k_indices] += h_k

    f_x_prime = f_fn(x_prime.reshape(1, -1)).item()

    C = max(y_prime, 1.0 - y_prime)
    return float(max(0.0, 1.0 - abs(f_x_prime - y_prime) / C))

## Helper Functions for LIME and SHAP Local Models

In [10]:
def lime_g_fn(local_exp_list, intercept, n_features):
    """Build local linear function from LIME output."""
    coef = np.zeros(n_features, dtype=np.float64)
    for item in local_exp_list:
        coef[item["feature_idx"]] = item["weight"]

    k_indices = np.array([item["feature_idx"] for item in local_exp_list], dtype=np.intp)
    weights   = np.array([item["weight"]      for item in local_exp_list], dtype=np.float64)
    bias      = float(intercept)

    def g(X):
        return np.asarray(X, dtype=np.float64) @ coef + bias

    return g, weights, k_indices


def shap_g_fn(shap_phi, expected_value, x, K):
    """Build local linear function from top-K SHAP values."""
    top_k = np.argsort(np.abs(shap_phi))[-K:].astype(np.intp)
    phi_k = shap_phi[top_k].astype(np.float64)

    intercept_s = float(expected_value) - float(np.dot(phi_k, x[top_k]))

    def g(X):
        Xarr = np.asarray(X, dtype=np.float64)
        if Xarr.ndim == 1:
            return intercept_s + float(np.dot(phi_k, Xarr[top_k]))
        return intercept_s + Xarr[:, top_k] @ phi_k

    return g, phi_k, top_k

## Reiteration Similarity Setup

We run each explainer R times per instance to measure how stable the selected feature set is.

In [11]:
rng_bg     = np.random.default_rng(SEED)
bg_indices = rng_bg.choice(X_train.shape[0], size=50, replace=False)
background = X_train[bg_indices]

shap_explainer = shap.KernelExplainer(
    lambda X: model.predict_proba(X)[:, 1], background
)


def shap_top_k_features(x, K, nsamples, seed):
    """Run SHAP once and return top-K feature indices."""
    np.random.seed(seed)
    sv  = np.array(shap_explainer.shap_values(x.reshape(1, -1), nsamples=nsamples)).flatten()
    return set(int(i) for i in np.argsort(np.abs(sv))[-K:])


def lime_ridge_top_k(x, K, H, seed):
    """Approximate LIME locally and return top-K feature indices."""
    rng   = np.random.default_rng(seed)
    sigma = np.std(X_train, axis=0) + 1e-9
    Z     = x + rng.normal(0, sigma, size=(H, len(x)))
    dists = np.linalg.norm(Z - x, axis=1)
    w     = np.exp(-(dists ** 2) / ((0.75 * np.sqrt(len(x))) ** 2))
    f_Z   = f_predict(Z)
    corr  = np.abs(np.corrcoef(Z.T, f_Z)[:-1, -1])
    return set(int(i) for i in np.argsort(corr)[-K:])


print("Explainers ready.")

Explainers ready.


## Compute All LEAF Metrics

In [12]:
results = []

for enum_i, idx in enumerate(test_indices):
    x   = X_test[idx]
    f_x = f_predict(x.reshape(1, -1)).item()

    lime_rec = lime_map[idx]
    lime_g, lime_w, lime_k_idx = lime_g_fn(
        lime_rec["local_exp"], lime_rec["intercept"], n_features
    )
    g_lime_x = float(lime_g(x))

    shap_phi = shap_values[enum_i]
    shap_g, shap_w, shap_k_idx = shap_g_fn(shap_phi, expected_value, x, K)
    g_shap_x = float(shap_g(x))

    conciseness = K

    fidelity_lime = local_fidelity(
        f_predict, lime_g, x, X_train, n_samples=N_PERTURB, seed=SEED + idx
    )
    fidelity_shap = local_fidelity(
        f_predict, shap_g, x, X_train, n_samples=N_PERTURB, seed=SEED + idx
    )

    concordance_lime = local_concordance(f_x, g_lime_x)
    concordance_shap = local_concordance(f_x, g_shap_x)

    lime_sets = [
        lime_ridge_top_k(x, K, NUM_SAMPLES, seed=SEED + r) for r in range(R)
    ]
    shap_sets = [
        shap_top_k_features(x, K, NUM_SAMPLES, seed=SEED + r) for r in range(R)
    ]
    reit_lime = reiteration_similarity(lime_sets)
    reit_shap = reiteration_similarity(shap_sets)

    shap_ic_s = float(expected_value) - float(np.dot(shap_w, x[shap_k_idx]))
    presc_lime = prescriptivity(f_predict, lime_w, lime_rec["intercept"], lime_k_idx, x)
    presc_shap = prescriptivity(f_predict, shap_w, shap_ic_s, shap_k_idx, x)

    results.append({
        "test_index":          int(idx),
        "true_y":              int(y_test[idx]),
        "f_x":                 f_x,
        "conciseness":         conciseness,
        "lime_fidelity":       fidelity_lime,
        "shap_fidelity":       fidelity_shap,
        "lime_concordance":    concordance_lime,
        "shap_concordance":    concordance_shap,
        "lime_reiteration":    reit_lime,
        "shap_reiteration":    reit_shap,
        "lime_prescriptivity": presc_lime,
        "shap_prescriptivity": presc_shap,
    })
    print(
        f"[{enum_i+1}/{len(test_indices)}] idx={idx}  "
        f"fid=({fidelity_lime:.2f},{fidelity_shap:.2f})  "
        f"conc=({concordance_lime:.2f},{concordance_shap:.2f})  "
        f"reit=({reit_lime:.2f},{reit_shap:.2f})  "
        f"presc=({presc_lime:.2f},{presc_shap:.2f})"
    )

print("\nDone.")

100%|██████████| 1/1 [00:00<00:00, 12.38it/s]


[1/10] idx=7  fid=(0.00,0.00)  conc=(0.00,0.40)  reit=(0.25,0.80)  presc=(0.00,0.00)


100%|██████████| 1/1 [00:00<00:00, 13.06it/s]


[2/10] idx=15  fid=(0.21,0.68)  conc=(0.38,0.62)  reit=(0.74,0.88)  presc=(0.61,0.00)


100%|██████████| 1/1 [00:00<00:00, 12.62it/s]


...

[8/10] idx=81  fid=(0.00,0.87)  conc=(0.00,0.60)  reit=(0.49,0.75)  presc=(0.04,0.00)


100%|██████████| 1/1 [00:00<00:00, 12.37it/s]


[9/10] idx=93  fid=(0.02,0.03)  conc=(0.00,0.40)  reit=(0.48,0.82)  presc=(0.00,0.00)


100%|██████████| 1/1 [00:00<00:00, 12.13it/s]

[10/10] idx=108  fid=(0.01,0.86)  conc=(0.00,0.60)  reit=(0.57,0.79)  presc=(0.00,0.00)

Done.


In [13]:
def avg(key):
    return float(np.mean([r[key] for r in results]))


summary = {
    "seed":       SEED,
    "K":          K,
    "R":          R,
    "n_perturb":  N_PERTURB,
    "n_examples": len(results),
    "avg": {
        "lime_fidelity":       avg("lime_fidelity"),
        "shap_fidelity":       avg("shap_fidelity"),
        "lime_concordance":    avg("lime_concordance"),
        "shap_concordance":    avg("shap_concordance"),
        "lime_reiteration":    avg("lime_reiteration"),
        "shap_reiteration":    avg("shap_reiteration"),
        "lime_prescriptivity": avg("lime_prescriptivity"),
        "shap_prescriptivity": avg("shap_prescriptivity"),
    },
    "results": results,
}

(OUT_DIR / "leaf_metrics.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("\n=== Average LEAF Metrics ===")
print(f"{'Metric':<25} {'LIME':>8} {'SHAP':>8}")
print("-" * 45)
for m in ["fidelity", "concordance", "reiteration", "prescriptivity"]:
    print(f"{m:<25} {summary['avg']['lime_'+m]:>8.3f} {summary['avg']['shap_'+m]:>8.3f}")


=== Average LEAF Metrics ===
Metric                        LIME     SHAP
---------------------------------------------
fidelity                     0.128    0.533
concordance                  0.136    0.523
reiteration                  0.572    0.811
prescriptivity               0.136    0.000


## Visualisation

In [14]:
metrics = [
    ("Local Fidelity",         "lime_fidelity",       "shap_fidelity"),
    ("Local Concordance",      "lime_concordance",    "shap_concordance"),
    ("Reiteration Similarity", "lime_reiteration",    "shap_reiteration"),
    ("Prescriptivity",         "lime_prescriptivity", "shap_prescriptivity"),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle(
    f"LEAF Metrics -- Breast Cancer Dataset, MLP Classifier (K={K}, R={R})",
    fontsize=13, fontweight="bold", y=1.02
)

for ax, (title, lime_key, shap_key) in zip(axes, metrics):
    lime_vals = [r[lime_key] for r in results]
    shap_vals = [r[shap_key] for r in results]

    bp = ax.boxplot(
        [lime_vals, shap_vals],
        patch_artist=True,
        medianprops={"color": "black", "linewidth": 2},
        widths=0.5,
    )
    bp["boxes"][0].set_facecolor("#4C72B0")
    bp["boxes"][1].set_facecolor("#DD8452")

    for j, vals in enumerate([lime_vals, shap_vals], start=1):
        ax.scatter(
            np.random.default_rng(j).uniform(j - 0.15, j + 0.15, len(vals)),
            vals, alpha=0.6, s=20,
            color=["#4C72B0", "#DD8452"][j - 1],
        )

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["LIME", "SHAP"])
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "leaf_metrics_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved leaf_metrics_boxplot.png")

Saved leaf_metrics_boxplot.png


/tmp/ipykernel_9299/857241624.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
metric_labels = ["Local Fidelity", "Local Concordance", "Reiteration Similarity", "Prescriptivity"]
lime_avgs = [avg("lime_fidelity"), avg("lime_concordance"), avg("lime_reiteration"), avg("lime_prescriptivity")]
shap_avgs = [avg("shap_fidelity"), avg("shap_concordance"), avg("shap_reiteration"), avg("shap_prescriptivity")]

x_pos = np.arange(len(metric_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x_pos - width/2, lime_avgs, width, label="LIME", color="#4C72B0")
ax.bar(x_pos + width/2, shap_avgs, width, label="SHAP", color="#DD8452")

ax.set_xticks(x_pos)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Average Score")
ax.set_title(f"Average LEAF Metrics -- LIME vs SHAP (K={K})")
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bar in ax.patches:
    ax.annotate(
        f"{bar.get_height():.2f}",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 3), textcoords="offset points",
        ha="center", va="bottom", fontsize=9
    )

plt.tight_layout()
plt.savefig(OUT_DIR / "leaf_metrics_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved leaf_metrics_bar.png")

Saved leaf_metrics_bar.png


/tmp/ipykernel_9299/3406505342.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
